> **⚠️ Deprecation Notice**
> This repository is no longer actively maintained. For the latest Gemma examples and tutorials, please visit the [Gemma Cookbook](https://github.com/google-gemma/cookbook).

##### Copyright 2025 Google LLC.

In [4]:
!nvidia-smi

Thu Oct  8 02:09:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import userdata
userdata.get('secretName')

In [2]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

In [ ]:
from google.colab import userdata
userdata.get('secretName')

# Multimodal understanding with Hugging Face Transformers

<table align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/google-gemini/gemma-cookbook/blob/main/Gemma/[Gemma_3n]Multimodal_understanding_with_HF.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
</table>

Gemma 3n is a generative AI model optimized for use in everyday devices, such as phones, laptops, and tablets.. This tutorial shows you how to get started running Gemma 3n using Hugging Face Transformers using Image + Audio input to generate text content. The Transformers Python library provides a API for accessing pre-trained generative AI models, including Gemma. For more information, see the [Transformers](https://huggingface.co/docs/transformers/en/index) documentation.

## Setup

Before starting this tutorial, complete the following steps:

* Get access to Gemma by logging into [Hugging Face](https://huggingface.co/google/gemma-3n-E4b-it) and selecting **Acknowledge license** for a Gemma model.
* Select a Colab runtime with sufficient resources to run
  the Gemma model size you want to run. [Learn more](https://ai.google.dev/gemma/docs/core#sizes).
* Generate a Hugging Face [Access Token](https://huggingface.co/docs/hub/en/security-tokens#how-to-manage-user-access-token) and use it to login from Colab.

This notebook will run on an NVIDIA T4 GPU using Gemma 3n E2B.\
But if you want to use Gemma 3n E4B, select L4 or A100.


In [9]:
from google.colab import userdata
from huggingface_hub import HfApi

token = userdata.get("HF_TOKEN")

usuario = HfApi(token=token).whoami()

print("Autenticado como:", usuario["name"])

Autenticado como: laisa10


In [12]:
!pip uninstall -y moviepy
!pip install -q -U "decorator>=5.2.1" "jedi>=0.16"

Found existing installation: moviepy 1.0.3
Uninstalling moviepy-1.0.3:
  Successfully uninstalled moviepy-1.0.3


In [13]:
!pip check

No broken requirements found.


In [17]:
from google.colab import userdata
from huggingface_hub import HfApi

token = userdata.get("HF_TOKEN")
api = HfApi(token=token)

print("Conta autenticada:", api.whoami()["name"])

Conta autenticada: laisa10


In [18]:
from huggingface_hub import hf_hub_download

arquivo = hf_hub_download(
    repo_id="google/gemma-3n-E2B-it",
    filename="config.json",
    token=userdata.get("HF_TOKEN")
)

print("Gemma autorizado!")

GatedRepoError: 403 Client Error. (Request ID: Root=1-6ac7009c-4e41e1a124edf7ce51aa0cae;d8b02800-b5d2-40fa-865b-de6148c3e295)

Cannot access gated repo for url https://huggingface.co/google/gemma-3n-E2B-it/resolve/main/config.json.
Your request to access model google/gemma-3n-E2B-it is awaiting a review from the repo authors.

In [14]:
import torch
import transformers
import librosa

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Librosa:", librosa.__version__)
print("GPU disponível:", torch.cuda.is_available())

PyTorch: 2.11.0+cu130
Transformers: 5.19.0
Librosa: 1.0.0
GPU disponível: True


In [16]:
from google.colab import userdata
from huggingface_hub import hf_hub_download

token = userdata.get("HF_TOKEN")
model_id = "google/gemma-3n-E2B-it"

hf_hub_download(
    repo_id=model_id,
    filename="config.json",
    token=token
)

print("Acesso ao Gemma 3n autorizado!")

GatedRepoError: 403 Client Error. (Request ID: Root=1-6ac6ffac-6e5f72c92f0728ce42117d18;cf11fb5c-e8fa-4eb6-8d40-70129b50af4f)

Cannot access gated repo for url https://huggingface.co/google/gemma-3n-E2B-it/resolve/main/config.json.
Access to model google/gemma-3n-E2B-it is restricted and you are not in the authorized list. Visit https://huggingface.co/google/gemma-3n-E2B-it to ask for access.

In [3]:
# Login into Hugging Face Hub
from huggingface_hub import notebook_login
notebook_login()

KeyboardInterrupt: 

### Install Python packages

Install the Hugging Face libraries required for running the Gemma model and making requests.

In [ ]:
# Install a transformers version that supports Gemma 3n (>= 4.53)
!pip install "transformers>=4.53.0" "timm>=1.0.16"

## Define formatting helper functions

Create a chat helper to manage and display the conversations.

In [ ]:
import torch

GEMMA_PATH = "google/gemma-3n-E2B-it" #@param ["google/gemma-3n-E2B-it", "google/gemma-3n-E4B-it"]
RESOURCE_URL_PREFIX = "https://raw.githubusercontent.com/google-gemini/gemma-cookbook/refs/heads/main/Demos/sample-data/"

from IPython.display import Audio, Image, Markdown, display

class ChatState():
  def __init__(self, model, processor):
    self.model = model
    self.processor = processor
    self.history = []

  def send_message(self, message, max_tokens=256):
    self.history.append(message)

    input_ids = self.processor.apply_chat_template(
        self.history,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )
    input_len = input_ids["input_ids"].shape[-1]

    input_ids = input_ids.to(self.model.device, dtype=model.dtype)
    outputs = self.model.generate(
        **input_ids,
        max_new_tokens=max_tokens,
        disable_compile=True
    )
    text = self.processor.batch_decode(
        outputs[:, input_len:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )
    self.history.append({
        "role": "assistant",
        "content": [
            {"type": "text", "text": text[0]},
        ]
    })

    # display chat
    for item in message['content']:
      if item['type'] == 'text':
        formatted_prompt = "<font size='+1' color='brown'>🙋‍♂️<blockquote>\n" + item['text'] + "\n</blockquote></font>"
        display(Markdown(formatted_prompt))
      elif item['type'] == 'audio':
        display(Audio(item['audio']))
      elif item['type'] == 'image':
        display(Image(item['image']))

    formatted_text = "<font size='+1' color='teal'>🤖<blockquote>\n" + text[0] + "\n</blockquote></font>"
    display(Markdown(formatted_text))


## Load Model

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor

processor = AutoProcessor.from_pretrained(GEMMA_PATH)
model = AutoModelForImageTextToText.from_pretrained(GEMMA_PATH, torch_dtype="auto", device_map="auto")

print(f"Device: {model.device}")
print(f"DType: {model.dtype}")

## Image-in: Captioning

In [ ]:
prompt = {
    "role": "user",
    "content": [
        {"type": "image", "image": f"{RESOURCE_URL_PREFIX}kitchen_painting.jpg"},
        {"type": "text", "text": "describe the image in detail"},
    ]
}

chat = ChatState(model, processor)
chat.send_message(prompt, max_tokens=512)

## Audio + Image : Tour Guide (image in + audio question)

This section demonstrates how to leverage Gemma 3n's multimodal input capability to assist as a Tour Guide. Take a picture of a famous landmark and ask any questions. Gemma 3n will act as your helpful tour guide.


In [ ]:
prompt = {
  "role": "user",
  "content": [
    {"type": "image", "image": f"{RESOURCE_URL_PREFIX}GoldenGate.png"},
    {"type": "audio", "audio": f"{RESOURCE_URL_PREFIX}tour_en.wav"},
  ]
}

chat = ChatState(model, processor)
chat.send_message(prompt)

## Try this by yourself

In [ ]:
!pip install ipywebrtc

Press the circle button and start speaking. Click the circle button again when you are finshed. The widget will immediately begin to play back what it captured.

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

from ipywebrtc import AudioRecorder, CameraStream

camera = CameraStream(constraints={'audio': True,'video':False})
recorder = AudioRecorder(stream=camera)
recorder

Convert webm file to wav format that PyTorch can understand.

In [ ]:
with open('/content/recording.webm', 'wb') as f:
    f.write(recorder.audio.value)
!ffmpeg -i /content/recording.webm /content/recording.wav -y

In [ ]:
prompt = {
  "role": "user",
  "content": [
    {"type": "image", "image": f"{RESOURCE_URL_PREFIX}GoldenGate.png"},
    {"type": "audio", "audio": "/content/recording.wav"},
  ]
}
chat = ChatState(model, processor)
chat.send_message(prompt)

## Next steps

Build and explore more with Gemma models:

* [Fine-tune Gemma for text tasks using Hugging Face Transformers](https://ai.google.dev/gemma/docs/core/huggingface_text_finetune_qlora)
* [Fine-tune Gemma for vision tasks using Hugging Face Transformers](https://ai.google.dev/gemma/docs/core/huggingface_vision_finetune_qlora)
* [Perform distributed fine-tuning and inference on Gemma models](https://ai.google.dev/gemma/docs/core/distributed_tuning)
* [Use Gemma open models with Vertex AI](https://cloud.google.com/vertex-ai/docs/generative-ai/open-models/use-gemma)
* [Fine-tune Gemma using Keras and deploy to Vertex AI](https://github.com/GoogleCloudPlatform/vertex-ai-samples/blob/main/notebooks/community/model_garden/model_garden_gemma_kerasnlp_to_vertexai.ipynb)